# 21. Chapter2 결과요약과 실험리포트

앞 노트북들이 저장한 실제 추론 결과를 모아 2장의 결론 리포트를 생성합니다.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch2_utils.py").exists():
    matches = (
        list(Path.cwd().glob("Deeplearning/*/2-1장/ch2_utils.py"))
        + list(Path.cwd().glob("Deeplearning/*/2장/ch2_utils.py"))
        + list(Path.cwd().glob("**/ch2_utils.py"))
    )
    NOTEBOOK_DIR = matches[0].parent if matches else Path("Deeplearning") / "Vision 응용" / "2-1장"
sys.path.append(str(NOTEBOOK_DIR))

from ch2_utils import *

paths = find_paths()
set_korean_font()
set_seed(7)
samples = load_samples(paths.data_root)
paths

## 21-1. 주요 결과 로드

In [ ]:
baseline_dir = paths.runs_root / "baseline_segformer_b0"
sample_metrics, group_metrics, class_metrics = load_run_metrics(baseline_dir)

files = {
    "exposure": paths.runs_root / "exposure_ratio_metrics_b0.csv",
    "hypothesis": paths.runs_root / "hypothesis_test_summary_b0.csv",
    "improvement": paths.runs_root / "improvement_strategy_summary_b0.csv",
    "interaction": paths.runs_root / "interaction_effects_b0.csv",
}
for name, path in files.items():
    print(name, path, path.exists())

## 21-2. 결과 요약 문장 생성

In [ ]:
report_lines = []
report_lines.append("# Chapter 2 실험 리포트")
report_lines.append("")
report_lines.append("## Baseline")
report_lines.append(conclusion_from_group(group_metrics, "color_group", "target_dice_mean"))
report_lines.append(conclusion_from_group(group_metrics, "defect_type", "target_dice_mean"))
report_lines.append(conclusion_from_group(group_metrics, "shape_group", "target_dice_mean"))

if files["interaction"].exists():
    interaction = pd.read_csv(files["interaction"])
    worst = interaction.sort_values("mean").iloc[0]
    report_lines.append("")
    report_lines.append("## Worst Interaction")
    report_lines.append(
        f"최저 조합은 {worst['color_group']} / {worst['shape_group']} / {worst['defect_type']}이며 Dice={worst['mean']:.3f}입니다."
    )

if files["exposure"].exists():
    exposure = pd.read_csv(files["exposure"])
    exposure_summary = exposure.groupby(["exposure_ratio", "eval_combo"])["target_dice"].mean().reset_index()
    deltas = []
    for combo, part in exposure_summary.groupby("eval_combo"):
        part = part.sort_values("exposure_ratio")
        deltas.append({"eval_combo": combo, "delta": part.iloc[-1]["target_dice"] - part.iloc[0]["target_dice"]})
    deltas = pd.DataFrame(deltas).sort_values("delta", ascending=False)
    report_lines.append("")
    report_lines.append("## Exposure Ratio")
    report_lines.append(
        f"red scratch 노출 비율 증가에 가장 크게 반응한 조합은 {deltas.iloc[0]['eval_combo']}이며 delta={deltas.iloc[0]['delta']:.3f}입니다."
    )

if files["hypothesis"].exists():
    hyp = pd.read_csv(files["hypothesis"])
    report_lines.append("")
    report_lines.append("## Hypothesis Tests")
    for _, row in hyp.iterrows():
        decision = "기각" if row["reject_h0_ci_excludes_0"] else "기각하지 않음"
        report_lines.append(
            f"- {row['group_col']} {row['group_a']} vs {row['group_b']}: diff={row['diff_a_minus_b']:.3f}, "
            f"CI=({row['ci95_low']:.3f}, {row['ci95_high']:.3f}) -> H0 {decision}"
        )

if files["improvement"].exists():
    imp = pd.read_csv(files["improvement"]).sort_values("worst_combo_dice", ascending=False)
    report_lines.append("")
    report_lines.append("## Improvement")
    report_lines.append(
        f"worst-combo 기준 최고 전략은 {imp.iloc[0]['strategy']}이며 Dice={imp.iloc[0]['worst_combo_dice']:.3f}입니다."
    )

report = "\n".join(report_lines)
out_path = paths.runs_root / "chapter2_report_b0.md"
out_path.write_text(report, encoding="utf-8")
print(report)
print("\n저장:", out_path)